# 08 · Comparativa de tasa de alertas a sensibilidad igualada

**Pregunta que responde:** igualando la sensibilidad de NEWS2, ¿el modelo alerta menos?
Es **la tesis del proyecto**, y este notebook es donde se contesta o se admite que todavía
no se puede contestar.

| | |
|---|---|
| **Issue** | `SCRUM-58` (usa el baseline de `SCRUM-56`) |
| **Notebook previo** | `07_news2_en_triage.ipynb`, que fijó la curva de NEWS2 |
| **Entrada** | `data/processed/triage-zenodo/triage.parquet` |
| **Cohorte** | TRIAGE (ADR-009), 1.300 pacientes de guardia, 54 muertes a 30 días |
| **Salida** | `data/interim/comparativa_sensibilidad_igualada.parquet` y figuras |
| **Semilla** | `20260930`, fija en `src/models/baseline.py` y en el bootstrap |

> ### El resultado, adelantado
> El modelo **discrimina mejor** que NEWS2 (AUROC 0,771 contra 0,728) y sobre el rango
> clínico reduce las alertas alrededor de un **30 %**. Pero el intervalo de confianza
> **incluye el cero**: con 54 eventos la reducción es *prometedora y no demostrada*.
> Decirlo de otra forma sería sobrevender el trabajo.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from src.data import alertas
from src.models import baseline
from src.viz import estilo

estilo.aplicar()
pd.set_option("display.width", 120)

INTERIM = RAIZ / "data" / "interim"
FIGURAS = INTERIM / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(RAIZ / "data" / "processed" / "triage-zenodo" / "triage.parquet")
df = df[df.puntuable]

resultado = baseline.evaluar_anidada(df, evento="muerte_30d")
y = resultado.y.astype(bool)
puntaje_modelo = resultado.puntajes
puntaje_news2 = df["news2_score"].astype(int).to_numpy(float)

print(f"pacientes: {len(df)}   muertes a 30 dias: {int(y.sum())} ({y.mean():.1%})")
print(f"familia elegida por cada fold externo: {', '.join(resultado.elegidos)}")
print()
print(f"AUROC modelo (CV anidada) : {roc_auc_score(y, puntaje_modelo):.3f}")
print(f"AUROC NEWS2               : {roc_auc_score(y, puntaje_news2):.3f}")

## 1 · Contra qué rival se compara

**Qué espero encontrar:** que las dos formas de usar NEWS2 den tasas distintas, y que la
regla desplegada sea la más débil de las dos.

Hay **dos** maneras de convertir NEWS2 en alertas, y elegir la equivocada cambia el
resultado:

- **La regla desplegada** — `risk_level != 'Bajo'`, la del trigger de la base. Incluye el
  rojo aislado.
- **El mejor umbral del score** — el corte sobre `news2_score` más barato que alcanza una
  sensibilidad dada.

El notebook 07 mostró que para mortalidad a 30 días **el mejor umbral le gana a la regla
desplegada**. Así que el rival honesto es el umbral, no la regla: comparar contra la regla
daría un número más lindo peleando contra el contrincante más flojo.

In [ ]:
alerta_desplegada = df["news2_riesgo"].isin(alertas.RIESGOS_QUE_ALERTAN).to_numpy()
sens_desplegada = float(alerta_desplegada[y].mean())
tasa_desplegada = float(alerta_desplegada.mean())

tasa_mejor_umbral = alertas.tasa_alertas_a_sensibilidad(y, puntaje_news2, sens_desplegada)

print(f"Sensibilidad de la regla desplegada: {sens_desplegada:.1%}")
print()
print(f"  NEWS2, regla desplegada : {tasa_desplegada:.1%} de alertas   (rival debil)")
print(f"  NEWS2, mejor umbral     : {tasa_mejor_umbral:.1%} de alertas   (rival REAL)")

## 2 · Las dos curvas, una sobre la otra

**Qué espero encontrar:** la curva del modelo por encima de la de NEWS2 en todo el rango —
más sensibilidad al mismo costo de alertas.

Ésta es la figura que resume el trabajo. Cada punto es un umbral posible; a la izquierda y
arriba está lo deseable: **detectar más, molestando menos**.

**Mirá el cruce cerca del 22 % de alertas.** Ahí el escalón de `score >= 4` de NEWS2 asoma
por encima de la curva del modelo. Es un tramo estrecho, pero es exactamente donde cae el
punto de operación del sistema desplegado — y es la razón de todo lo que viene en la §3.

In [ ]:
curva_news2 = alertas.curva_operacion(y, puntaje_news2)
curva_modelo = alertas.curva_operacion(y, puntaje_modelo)

fig, ax = plt.subplots(figsize=(8.8, 4.8))

ax.plot(100 * curva_modelo.tasa_alertas, 100 * curva_modelo.sensibilidad,
        color=estilo.TERRACOTA, linewidth=2.4, zorder=3, label="Modelo (CV anidada)")
ax.plot(100 * curva_news2.tasa_alertas, 100 * curva_news2.sensibilidad,
        color=estilo.GRIS, linewidth=2, linestyle=(0, (5, 3)), marker="o",
        markersize=5, markerfacecolor=estilo.SUPERFICIE, zorder=2, label="NEWS2 (score)")

ax.scatter([100 * tasa_desplegada], [100 * sens_desplegada], s=140, marker="D",
           color=estilo.OLIVA, zorder=4, edgecolor=estilo.SUPERFICIE, linewidth=1.5)
ax.annotate("regla desplegada", xy=(100 * tasa_desplegada, 100 * sens_desplegada),
            xytext=(14, -18), textcoords="offset points", fontsize=9, color=estilo.OLIVA,
            arrowprops=dict(arrowstyle="-", color=estilo.OLIVA, linewidth=1,
                            shrinkA=0, shrinkB=6))

# La banda donde se compara. Fuera de ella la comparacion no es clinicamente util.
ax.axhspan(50, 80, color=estilo.GRIS_TENUE, alpha=0.35, zorder=0)
ax.annotate("rango donde se compara\n(50-80 % de sensibilidad)", xy=(97, 65),
            ha="right", va="center", fontsize=8.5, color=estilo.GRIS, linespacing=1.4)

ax.set_xlabel("Tasa de alertas (% de admisiones)")
ax.set_ylabel("Sensibilidad para muerte a 30 dias (%)")
ax.set_title("El modelo domina en casi todo el rango — pero las curvas se cruzan cerca del 22 %",
             loc="left", color=estilo.TINTA, fontsize=10.5)
ax.set_xlim(-3, 103)
ax.set_ylim(-3, 103)
ax.legend(loc="lower right")
fig.savefig(FIGURAS / "08_curvas_superpuestas.png")
plt.show()

## 3 · Por qué NO se compara en un punto único

**Éste es el hallazgo metodológico del notebook**, y conviene entenderlo antes de mirar
cualquier número.

El score de NEWS2 es **entero y con pocos niveles efectivos**, así que su curva de
operación avanza a **escalones grandes**: entre `score >= 4` y `score >= 5` la tasa de
alertas salta del 22,6 % al 14,4 % de un tirón. Comparar en una sensibilidad única cae,
según la suerte, justo antes o justo después de un escalón — y el resultado se mueve
muchísimo sin que haya cambiado nada del modelo.

**Qué espero encontrar:** que la reducción medida en el punto donde opera hoy el sistema
sea mucho menor que el promedio sobre el rango, porque ese punto cae justo después de un
escalón favorable a NEWS2.

In [ ]:
punto_unico = 1 - alertas.tasa_alertas_a_sensibilidad(
    y, puntaje_modelo, sens_desplegada
) / tasa_mejor_umbral

filas = []
for objetivo in alertas.GRILLA_SENSIBILIDAD:
    a = alertas.tasa_alertas_a_sensibilidad(y, puntaje_news2, objetivo)
    b = alertas.tasa_alertas_a_sensibilidad(y, puntaje_modelo, objetivo)
    filas.append({
        "sensibilidad": f"{objetivo:.0%}",
        "NEWS2": f"{a:.1%}",
        "modelo": f"{b:.1%}",
        "reduccion": f"{1 - b / a:+.1%}",
    })
tabla = pd.DataFrame(filas).set_index("sensibilidad")
print(tabla.to_string())
print()
print(f"En el punto unico donde opera hoy el sistema ({sens_desplegada:.1%}): {punto_unico:+.1%}")
print(f"Promedio sobre el rango 50-80%                       : "
      f"{np.mean([float(f['reduccion'].strip('%+')) / 100 for f in filas]):+.1%}")

**Lo que se ve:** el punto único da una reducción chica y el promedio del rango, una
grande. **No es que un número sea el verdadero y el otro mentira**: es que el punto único
**no es un estimador estable** cuando el rival tiene una escala discreta y gruesa. Por eso
`SCRUM-58` se reporta sobre el rango, y la decisión está escrita en
`alertas.reduccion_relativa`.

El rango 50-80 % tampoco es arbitrario: por debajo del 50 % una escala de alerta temprana
no cumple su función, y por encima del 80 % la tasa de alertas se dispara hasta volverse
inaplicable en una guardia real.

## 4 · El número, con su intervalo

**Qué espero encontrar:** una reducción media sustancial y un intervalo ancho que roce o
cruce el cero, porque 54 eventos son pocos.

Sin intervalo, cualquier porcentaje acá sería precisión falsa.

In [ ]:
punto, lo, hi = alertas.reduccion_relativa(y, puntaje_news2, puntaje_modelo)

print(f"Reduccion relativa media de la tasa de alertas (sensibilidad 50-80 %)")
print(f"  punto  : {punto:+.1%}")
print(f"  IC 95 %: [{lo:+.1%}, {hi:+.1%}]")
print()
if lo > 0:
    print("  El intervalo EXCLUYE el cero: la reduccion esta demostrada.")
else:
    print("  El intervalo INCLUYE el cero: la reduccion NO esta demostrada.")
    print("  El modelo es prometedor —discrimina mejor— pero con esta cantidad de")
    print("  eventos no se puede afirmar que reduzca las alertas.")

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 2.6))

ax.axvline(0, color=estilo.GRIS, linewidth=1.5, linestyle=(0, (4, 3)), zorder=1)
ax.plot([100 * lo, 100 * hi], [0, 0], color=estilo.TERRACOTA, linewidth=6,
        solid_capstyle="round", zorder=2)
ax.scatter([100 * punto], [0], s=120, color=estilo.TINTA, zorder=3)

ax.annotate(f"{punto:+.1%}", xy=(100 * punto, 0), xytext=(0, 14),
            textcoords="offset points", ha="center", fontsize=10.5, color=estilo.TINTA)
for valor, ha, dx in ((lo, "right", -6), (hi, "left", 6)):
    ax.annotate(f"{valor:+.1%}", xy=(100 * valor, 0), xytext=(dx, -16),
                textcoords="offset points", ha=ha, fontsize=9, color=estilo.GRIS)
ax.annotate("sin efecto", xy=(0, 0), xytext=(0, -26), textcoords="offset points",
            ha="center", fontsize=9, color=estilo.GRIS)

ax.set_yticks([])
ax.set_xlabel("Reduccion relativa de la tasa de alertas (%)")
ax.set_title("El intervalo cruza el cero: prometedor, no demostrado",
             loc="left", color=estilo.TINTA)
ax.grid(axis="y", visible=False)
ax.margins(y=0.6)
fig.savefig(FIGURAS / "08_intervalo_reduccion.png")
plt.show()

## 5 · Cuántos datos harían falta

**Qué espero encontrar:** que el intervalo se angoste al crecer la cohorte, y poder decir
a partir de qué tamaño dejaría de cruzar el cero.

Se remuestrea la cohorte con reemplazo a tamaños múltiplos del actual. **El supuesto es
fuerte y hay que declararlo:** esto asume que la muestra representa a la población y que
el efecto observado es el real. No reemplaza a un estudio de potencia hecho como
corresponde, pero da el orden de magnitud.

In [ ]:
def reduccion_en(indices):
    yy = y[indices]
    rs = []
    for objetivo in alertas.GRILLA_SENSIBILIDAD:
        a = alertas.tasa_alertas_a_sensibilidad(yy, puntaje_news2[indices], objetivo)
        b = alertas.tasa_alertas_a_sensibilidad(yy, puntaje_modelo[indices], objetivo)
        if a and a == a and b == b:
            rs.append(1 - b / a)
    return np.mean(rs) if rs else np.nan

rng = np.random.default_rng(20260930)
n = len(y)
filas = []
for mult in (1, 2, 5, 10, 20):
    reps = np.array([reduccion_en(rng.integers(0, n, n * mult)) for _ in range(400)])
    v = reps[~np.isnan(reps)]
    bajo, alto = np.percentile(v, [2.5, 97.5])
    filas.append({
        "cohorte": f"{mult}x",
        "pacientes": n * mult,
        "eventos": int(y.sum() * mult),
        "IC 95 %": f"[{bajo:+.1%}, {alto:+.1%}]",
        "excluye 0": "SI" if bajo > 0 else "no",
    })

potencia = pd.DataFrame(filas).set_index("cohorte")
print(potencia.to_string())

**La lectura, y es mejor noticia de lo esperado:** el intervalo deja de cruzar el cero
ya en **2x — unos 108 eventos, del orden de 2.600 pacientes**. No hacen falta cientos de
miles: hace falta **aproximadamente el doble** de la cohorte actual.

Eso cambia la conversación. La base completa de MIMIC-IV-ED tiene ~425.000 episodios, o
sea **mucho más de lo necesario** para este resultado concreto; con una fracción alcanza.

Dicho de otro modo: **el cuello de botella de este proyecto no es el modelo, es la
cohorte** — pero es un cuello angosto y salvable, no un abismo. `SCRUM-48` no es un
trámite administrativo pendiente: es la condición para que la afirmación central de la
tesis pase de "consistente" a "demostrada".

> ⚠️ **El supuesto de esta simulación es fuerte**: remuestrear con reemplazo asume que la
> muestra representa a la población y que el efecto observado —el 29,8 %— es el real. Si
> el efecto verdadero fuera la mitad, harían falta cerca de cuatro veces más eventos. Es
> un orden de magnitud, no un cálculo de potencia formal.

In [ ]:
salida = pd.DataFrame([{
    "cohorte": "triage-zenodo",
    "evento": "muerte_30d",
    "n": len(y),
    "eventos": int(y.sum()),
    "auroc_modelo": round(roc_auc_score(y, puntaje_modelo), 4),
    "auroc_news2": round(roc_auc_score(y, puntaje_news2), 4),
    "grilla_sensibilidad": " | ".join(f"{g:.0%}" for g in alertas.GRILLA_SENSIBILIDAD),
    "reduccion_media": round(punto, 4),
    "ic95_bajo": round(lo, 4),
    "ic95_alto": round(hi, 4),
    "demostrada": bool(lo > 0),
    "reduccion_en_punto_unico": round(punto_unico, 4),
    "sensibilidad_del_sistema_desplegado": round(sens_desplegada, 4),
}])
salida.to_parquet(INTERIM / "comparativa_sensibilidad_igualada.parquet", index=False)
print(salida.T.to_string(header=False))

## Conclusiones

**El resultado, redactado como va a la tesis:**

> Sobre la cohorte TRIAGE (1.300 pacientes de guardia, 54 muertes a 30 días), un modelo
> entrenado únicamente con los siete parámetros disponibles al pie de la cama discrimina
> mejor que NEWS2 (**AUROC 0,771 contra 0,728**, con validación cruzada anidada) y, sobre
> el rango de sensibilidad clínicamente útil (50-80 %), reduce la tasa de alertas en un
> **29,8 %** de media. **El intervalo de confianza del 95 % incluye el cero**, de modo que
> la reducción es consistente pero no estadísticamente demostrada con esta cohorte.

**Qué se concluyó**

1. **El modelo discrimina mejor que NEWS2**, y el estimador es insesgado: la selección de
   familia ocurre dentro del bucle de evaluación.
2. **La reducción de alertas es sustancial pero no demostrada.** Decir "reduce un 30 % las
   alertas" sin el intervalo sería sobrevender el trabajo.
3. **Comparar en un punto único es un error metodológico** cuando el rival tiene una escala
   discreta y gruesa: en el punto donde opera hoy el sistema la reducción da `+2,7 %` y
   sobre el rango da `+29,8 %`, sin que nada del modelo haya cambiado.
4. **El rival correcto es el mejor umbral de score**, no la regla desplegada, que es más
   débil. Comparar contra la regla inflaría el resultado.
5. **El límite es la cohorte, no el modelo — y está cerca.** La simulación indica que con
   **el doble de la cohorte actual** (~108 eventos, ~2.600 pacientes) el intervalo ya
   excluiría el cero. MIMIC-IV-ED completo tiene ~425.000 episodios, así que sobra.

**Qué queda abierto**

- **Re-correr este notebook sobre MIMIC completo** cuando llegue la credencial
  (`SCRUM-48`). No cambia una línea de código.
- **`SCRUM-59`** — exportar a ONNX el modelo elegido, que resultó ser la regresión
  logística y no XGBoost.
- **Enmendar el ADR-005**, que da por sentado que XGBoost es el modelo principal.
- **`SCRUM-57`** (LSTM) sigue bloqueada: esta cohorte no tiene series temporales.